# Composition Agent (multi-tool chaining test)

Derived from `interactive_agent.ipynb` — **do not edit that notebook**. This notebook is a focused test of **multi-step tool composition**: the LLM must call `seqmagick_info` first, then chain to `seqmagick_convert` (reverse complement) using the result/context from step 1. Retrieval offers both tools (top_k=20); the run loop feeds each tool result back so the LLM can pick the next step.

Default query is a two-step task. Run all cells in order (Colab `git pull` in cell 1 then execute sequentially). Watch for:
- Iteration 1 → `seqmagick_info` (auto-install + execute)
- Iteration 2 → `seqmagick_convert` with `reverse_complement: true` (chained, not single-shot)
- Final answer combining both steps.


# Tool-Discovery Agent

Discover bioinformatics tools from papers, inject them into your agent, and run tasks.

**Flow:** Scan agent -> Inject tools -> Run tasks via downstream agent

In [ ]:
import os, sys, subprocess
ST2_DIR = '/content/st2'
if os.path.isdir(os.path.join(ST2_DIR, '.git')):
    subprocess.run(['git', '-C', ST2_DIR, 'pull', '--ff-only'], capture_output=True)
else:
    if os.path.exists(ST2_DIR):
        subprocess.run(['rm', '-rf', ST2_DIR], check=True)
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/caixiaoyao2025/st2.git', ST2_DIR], check=True)
ST2_DIR = '/content/st2'
sys.path.insert(0, ST2_DIR)
os.chdir(ST2_DIR)
# Demo input so the default task runs through without the user
# supplying a file. seqmagick is pure-Python/pip-installable, so it
# works in Colab; bqtools would need a Rust/cargo toolchain.
_DEMO_FASTA = ('>seq1 example\n'
              'ACGTACGTACGTACGTACGT\n'
              '>seq2 example\n'
              'TTGGCCTAAGGCCTTAGGCA\n'
              '>seq3 example\n'
              'GATTACAGATTACAGATTACA\n')
with open('reads.fasta', 'w') as _f:
    _f.write(_DEMO_FASTA)
print('demo reads.fasta written:', len(_DEMO_FASTA), 'bytes')
!pip install -q langchain langchain-openai langgraph pydantic graph-tool-call 2>/dev/null || true
from IPython.display import display, Markdown
display(Markdown('**Setup done**'))

## Step 1 - Input your agent & API key

In [ ]:
import os, subprocess
from IPython.display import display, Markdown
import ipywidgets as widgets

# ---- Known agent mappings: method + register method ----
KNOWN_AGENTS = {
    'biomni': {'method': 'go', 'register': 'add_tool'},
    'cellagent': {'method': 'run', 'register': 'add_tool'},
    'geneagent': {'method': 'run', 'register': 'add_tool'},
    'crispr': {'method': 'run', 'register': 'add_tool'},
    'biochatter': {'method': 'run', 'register': 'add_tool'},
    'langchain': {'method': 'invoke', 'register': 'add_tool'},
    'smolagents': {'method': 'run', 'register': 'add_tool'},
    'dspy': {'method': 'forward', 'register': 'add_tool'},
    'crewai': {'method': 'kickoff', 'register': 'add_tool'},
    'metagpt': {'method': 'run', 'register': 'add_tool'},
}
PROBE_ORDER = ['go', 'run', 'execute', 'predict', 'forward', 'invoke']

# ---- Agent presets: pick from the dropdown to fill the URL and auto-connect ----
AGENT_PRESETS = {
    'Biomni A1 (snap-stanford/Biomni)': 'https://github.com/snap-stanford/Biomni.git',
    'BioChatter (biocypher/biochatter)': 'https://github.com/biocypher/biochatter.git',
    'smolagents (huggingface)': 'https://github.com/huggingface/smolagents.git',
    'DSPy (stanfordnlp)': 'https://github.com/stanfordnlp/dspy.git',
    'CrewAI': 'https://github.com/crewAIInc/crewAI.git',
    'MetaGPT': 'https://github.com/geekan/MetaGPT.git',
}
agent_dropdown = widgets.Dropdown(
    options=['(custom URL)'] + list(AGENT_PRESETS.keys()),
    value='Biomni A1 (snap-stanford/Biomni)',
    description='Agent:', layout=widgets.Layout(width='90%'))

path_input = widgets.Text(
    value='https://github.com/snap-stanford/Biomni.git',
    placeholder='Git URL or /content/my_agent',
    description='Agent:', layout=widgets.Layout(width='90%'))
key_input = widgets.Password(
    value='sk-CXVEKD43upOkHWTdq1RJP3SMC4OyspQOkqB4ymqw6IJazWyB', placeholder='ark-... or sk-...',
    description='API Key:', layout=widgets.Layout(width='90%'))
model_input = widgets.Text(
    value='hy3',
    description='Model:', layout=widgets.Layout(width='70%'))
base_input = widgets.Text(
    value='https://tokenhub.tencentmaas.com/v1',
    description='Base URL:', layout=widgets.Layout(width='90%'))
btn = widgets.Button(description='Connect agent', button_style='primary')
out = widgets.Output()

def on_connect(_):
    out.clear_output()
    with out:
        agent_src = path_input.value.strip()
        api_key = key_input.value.strip()
        model = model_input.value.strip()
        base_url = base_input.value.strip()
        if not agent_src:
            display(Markdown('**ERROR:** Enter agent path or URL')); return
        if not api_key:
            display(Markdown('**ERROR:** Enter API key')); return
        if agent_src.startswith('http'):
            agent_dir = '/content/_agent_' + agent_src.split('/')[-1].replace('.git', '')
            if not os.path.isdir(agent_dir):
                subprocess.run(['git', 'clone', '--depth', '1', agent_src, agent_dir],
                               capture_output=True, text=True)
            display(Markdown(f'Cloned to `{agent_dir}`'))
        else:
            agent_dir = agent_src
            if not os.path.isdir(agent_dir):
                display(Markdown(f'**ERROR:** `{agent_dir}` not found')); return
        os.environ['OPENAI_API_KEY'] = api_key
        os.environ['OPENAI_BASE_URL'] = base_url
        os.environ['OPENAI_MODEL'] = model
        os.environ['WESTLAKE_API_KEY'] = api_key
        os.environ['BIOMNI_SOURCE'] = 'Custom'
        os.environ['BIOMNI_LLM'] = model
        os.environ['BIOMNI_CUSTOM_BASE_URL'] = base_url
        os.environ['BIOMNI_CUSTOM_API_KEY'] = api_key
        get_ipython().user_ns['agent_dir'] = agent_dir
        get_ipython().user_ns['api_key_val'] = api_key
        get_ipython().user_ns['model_val'] = model
        get_ipython().user_ns['base_url_val'] = base_url
        display(Markdown(f'**Agent:** `{agent_dir}` | **Model:** `{model}` | **Key:** `{api_key[:8]}...`'))

def _on_preset_change(change):
    if change.get('name') != 'value' or change.get('new') == '(custom URL)':
        return
    url = AGENT_PRESETS.get(change['new'])
    if url:
        path_input.value = url
        on_connect(None)

agent_dropdown.observe(_on_preset_change)
btn.on_click(on_connect)
display(widgets.VBox([agent_dropdown, path_input, key_input, model_input, base_input, btn, out]))

## Step 2 - Scan agent & detect wiring

In [ ]:
import os, sys, json
from IPython.display import display, Markdown
import yaml

from agent_connector.scanner import build_schema

schema = build_schema(agent_dir, include_evidence=False)
detected = [
    '**Detected:**',
    '- agent_class = `' + str(schema.get('agent_class', 'N/A')) + '`',
    '- module_path = `' + str(schema.get('module_path', 'N/A')) + '`',
    '- registration_method = `' + str(schema.get('registration_method', 'N/A')) + '`',
    '- wiring_style = `' + str(schema.get('wiring_style', 'N/A')) + '`',
    '- init_signature = `' + str(schema.get('init_signature', 'N/A')) + '`',
]
display(Markdown('<br>'.join(detected)))

reg_path = os.path.join(ST2_DIR, 'data', 'mcp_registry.yaml')
tools = yaml.safe_load(open(reg_path, encoding='utf-8'))['tools']
display(Markdown(f'**Registry:** {len(tools)} tools'))

## Step 3 - Resolve execution method

In [ ]:
from IPython.display import display, Markdown

agent_class = (schema.get('agent_class') or '').lower()

# Layer 1: known agent -> direct mapping
exec_method = None
for pat, info in KNOWN_AGENTS.items():
    if pat in agent_class or pat in agent_dir.lower():
        exec_method = info['method']
        display(Markdown(f'**Known agent** `{pat}` -> `{exec_method}`'))
        break

# Layer 2: scan source for .go()/.run() hints
if exec_method is None:
    hits = {}
    for root, dirs, files in os.walk(agent_dir):
        dirs[:] = [d for d in dirs if d not in ('.git','__pycache__','node_modules','.venv')]
        for f in files:
            if f.endswith('.py'):
                try:
                    text = open(os.path.join(root,f), encoding='utf-8', errors='replace').read()
                except: continue
                for m in PROBE_ORDER:
                    hits[m] = hits.get(m, 0) + text.count(f'.{m}(')
    if any(v > 0 for v in hits.values()):
        exec_method = max(hits, key=hits.get)
        display(Markdown(f'**Source hint** -> `{exec_method}` ({hits[exec_method]} occurrences)'))

# Layer 3: default
if exec_method is None:
    exec_method = 'run'
    display(Markdown('**No signal found.** Defaulting to `run`.'))

get_ipython().user_ns['exec_method_override'] = exec_method

## Step 4 - Preflight check & install

In [ ]:
import os, sys, subprocess as _sp
from IPython.display import display, Markdown
from agent_connector.agent_preflight import preflight, preflight_report

# Auto-install missing agent runtime deps (Layer 1) detected by the preflight.
pf = preflight(agent_dir, agent_module_path=schema.get('module_path'), install_missing=True)

# Most OpenAI-compatible agents (incl. BioChatter) also need the LLM backend at
# import time. Layer 2 tool-deps are NOT auto-installed (some are import-tracer
# artifacts like 'base'/'web' that are not real packages), so install the
# known-safe LLM-backend subset explicitly.
for _pkg in ['openai', 'langchain-openai', 'pydantic', 'langchain-community',
             'langchain-mcp-adapters', 'nest_asyncio', 'mcp', 'fastmcp', 'pyyaml']:
    try:
        _r = _sp.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg],
                     capture_output=True, text=True, timeout=600)
        if _r.returncode != 0:
            print(f'  (pip install {_pkg} note: {_r.stderr.strip()[:200]})')
    except Exception as _e:
        print(f'  pip install {_pkg} failed: {_e}')

# BioChatter reads its OpenAI-compatible endpoint from GENERIC_TEST_OPENAI_BASE_URL.
_mod = (schema.get('module_path') or '').lower()
if 'biochatter' in agent_dir.lower() or 'biochatter' in _mod:
    os.environ['GENERIC_TEST_OPENAI_BASE_URL'] = os.environ.get('OPENAI_BASE_URL', '')
    display(Markdown('**BioChatter env:** `GENERIC_TEST_OPENAI_BASE_URL` set to the OpenAI base URL.'))

display(Markdown(preflight_report(pf)))


In [ ]:
import subprocess, os, sys
from IPython.display import display, Markdown

if pf.status == 'SETUP_REQUIRED':
    pkgs = pf.pip_installable
    if pkgs:
        display(Markdown(f'Installing **{len(pkgs)}** packages...'))
        cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + pkgs
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
        display(Markdown(f'**Done** (returncode={r.returncode})'))
    else:
        display(Markdown('No auto-installable packages'))
else:
    display(Markdown(f'Status = `{pf.status}`'))

## Step 5 - Create agent & inject tools

In [ ]:
import os, sys, json, importlib
from IPython.display import display, Markdown

from agent_connector.generator import generate_wiring, load_wrappers, load_adapter

schema['execution_method'] = exec_method_override

import ipywidgets as _widgets
# Agent is auto-detected from the connected repo via build_schema() -- no manual
# selection. Exec mode only chooses how tools are driven: 'Unified (Tool Runner)'
# is agent-agnostic and recommended; 'Native' lets react/biochatter self-drive.
exec_mode = _widgets.Dropdown(options=['Unified (recommended)', 'Native tool calling'],
                              value='Unified (recommended)', description='Exec mode:')
display(exec_mode)


# Force-set BIOMNI env vars BEFORE any agent import (so default_config picks them up)
import os as _os
_m = _os.environ.get('OPENAI_MODEL', '')
_k = _os.environ.get('OPENAI_API_KEY') or _os.environ.get('WESTLAKE_API_KEY')
_u = _os.environ.get('OPENAI_BASE_URL', '')
if _m and not _os.environ.get('BIOMNI_LLM'): _os.environ['BIOMNI_LLM'] = _m
if _k and not _os.environ.get('BIOMNI_CUSTOM_API_KEY'): _os.environ['BIOMNI_CUSTOM_API_KEY'] = _k
if _u and not _os.environ.get('BIOMNI_CUSTOM_BASE_URL'): _os.environ['BIOMNI_CUSTOM_BASE_URL'] = _u
if not _os.environ.get('BIOMNI_SOURCE'): _os.environ['BIOMNI_SOURCE'] = 'Custom'

# Generate wiring
wiring_dir = os.path.join(ST2_DIR, 'wiring')
wiring = generate_wiring(tools, schema, out_dir=wiring_dir)
display(Markdown('Wiring mode: `' + wiring['mode'] + '`'))

# Load wrappers as functions (Biomni needs inspect.getsource)
sys.path.insert(0, wiring_dir)
sys.path.insert(0, agent_dir)
wrappers = load_wrappers(package_name='generated_tools', registration_style='function')
display(Markdown(f'**{len(wrappers)} wrappers loaded**'))

# Create agent via adapter
agent = None
adapter = None

if schema.get('module_path') or schema.get('registration_method'):
    try:
        _adapter_path = wiring['artifacts'].get('adapter')
        adapter = load_adapter(schema.get('agent_class') or 'Agent',
                               adapter_path=_adapter_path) if _adapter_path else None
        agent = adapter.create_agent(use_tool_retriever=False)
        display(Markdown(f'**Agent created via adapter:** `{type(agent).__name__}`'))
    except Exception as e:
        display(Markdown(f'`create_agent()` failed: `{e}`'))

# Fallback: if adapter.create_agent() failed (e.g. react needs all Biomni deps), try A1
# A1 is lighter — it doesn't load all built-in tools in __init__
if agent is None and 'biomni' in agent_dir.lower():
    try:
        sys.path.insert(0, agent_dir)
        from biomni.agent.a1 import A1
        agent = A1(
            path=os.path.join(ST2_DIR, '_agent_data'),
            llm=os.environ.get('OPENAI_MODEL', 'gpt-4o'),
            source='Custom',
            base_url=os.environ.get('OPENAI_BASE_URL'),
            api_key=os.environ.get('OPENAI_API_KEY'),
            use_tool_retriever=False,
            expected_data_lake_files=[],
        )
        display(Markdown(f'**A1 agent created:** `{type(agent).__name__}`'))
    except Exception as e:
        display(Markdown(f'A1 fallback failed: `{e}`'))

# Fallback: DynamicAgent
if agent is None:
    class DynamicAgent:
        def __init__(self): self.tools = []
    DynamicAgent.add_tool = lambda self, t: self.tools.append(t)
    agent = DynamicAgent()
    display(Markdown(f'**Fallback:** DynamicAgent'))

# Inject tools: create LangChain tools from _TOOL_SPEC and append to agent.tools
from langchain_core.tools import tool as lc_tool
from agent_connector.tool_runner import run_tool_spec, format_result

# Deterministic schema generation for A1's add_tool: building the API
# schema via the LLM is flaky (truncation / 'not applicable') AND burns the
# LLM quota (402 free-trial exhaustion). Our wrappers carry a _TOOL_SPEC,
# so build the schema directly from it -- no LLM call.
try:
    import ast as _ast
    import biomni.agent.a1 as _a1mod
    _orig_ftas = _a1mod.function_to_api_schema
    # Only patch A1's schema generator (the only agent whose add_tool burns
    # the LLM on a response_format deepseek/TokenHub rejects). For other
    # agents the unified loop executes tools via spec_lookup, so add_tool
    # schema quality is irrelevant.
    _PATCH_A1 = 'a1' in (schema.get('agent_class') or '').lower()
    if not _PATCH_A1:
        raise RuntimeError('skip A1 monkeypatch for non-A1 agent')

    _SPEC_BY_NAME = {}
    try:
        import sys as _sys
        for _w in wrappers:
            _n = getattr(_w, "__name__", None)
            _mod = _sys.modules.get(getattr(_w, "__module__", "")) if _n else None
            _s = getattr(_mod, "_TOOL_SPEC", None) if _mod else None
            if _s is None:
                _s = getattr(_w, "_TOOL_SPEC", None)
            if _n and _s:
                _SPEC_BY_NAME[_n] = _s
                _sn = _s.get("name")
                if _sn:
                    _SPEC_BY_NAME[_sn] = _s
    except Exception:
        pass

    def _reliable_function_to_api_schema(function_string, llm):
        _spec = None
        try:
            _tree = _ast.parse(function_string)
            for _node in _tree.body:
                if isinstance(_node, _ast.Assign):
                    for _t in _node.targets:
                        if isinstance(_t, _ast.Name) and _t.id == '_TOOL_SPEC':
                            _spec = _ast.literal_eval(_node.value)
                elif isinstance(_node, (_ast.FunctionDef, _ast.ClassDef)) and _spec is None:
                    _spec = _SPEC_BY_NAME.get(_node.name)
        except Exception:
            _spec = None
        if _spec is None:
            try:
                _t2 = _ast.parse(function_string)
                for _node in _t2.body:
                    if isinstance(_node, (_ast.FunctionDef, _ast.ClassDef)):
                        _spec = _SPEC_BY_NAME.get(_node.name)
                        if _spec:
                            break
            except Exception:
                _spec = None
        if _spec is not None:
            _req, _opt = [], []
            for _pname, _meta in (_spec.get('inputs') or {}).items():
                if _pname == 'subcommand':
                    continue
                _m = _meta or {}
                _entry = {'name': _pname, 'type': 'str',
                          'description': _m.get('description', ''), 'default': None}
                (_req if _m.get('required') else _opt).append(_entry)
            return {'name': _spec.get('name'),
                    'description': _spec.get('description', ''),
                    'required_parameters': _req,
                    'optional_parameters': _opt}
        return _orig_ftas(function_string, llm)

    _a1mod.function_to_api_schema = _reliable_function_to_api_schema
except Exception:
    pass

if not hasattr(agent, 'tools') or agent.tools is None:
    agent.tools = []

injected = 0
# Code-execution agents (e.g. A1) expose tools to their Python REPL via
# agent.add_tool(fn), which registers fn into builtins._biomni_custom_functions
# -- the ONLY namespace A1's <execute> blocks can see. Appending a LangChain
# lc_tool to agent.tools does NOT make it callable inside A1. So for agents
# that have add_tool(), also register the RAW wrapper function there.
_register_via_add_tool = hasattr(agent, 'add_tool')
for w in wrappers:
    ts = getattr(w, '_TOOL_SPEC', None)
    if ts is None and hasattr(w, '__module__'):
        mod = importlib.import_module(w.__module__)
        ts = getattr(mod, '_TOOL_SPEC', None)
    if ts:
        def _make_runner(spec=ts):
            _desc = spec.get('description', spec['name'])
            # Build input schema string for LLM
            _inputs = spec.get('inputs') or {}
            if _inputs:
                _params = []
                for _pname, _pmeta in _inputs.items():
                    _req = 'required' if (_pmeta or {}).get('required') else 'optional'
                    _ptype = (_pmeta or {}).get('type', 'string')
                    _pdesc = (_pmeta or {}).get('description', '')
                    _params.append(f'  {_pname} ({_ptype}, {_req}): {_pdesc}')
                _desc += '\nParameters:\n' + chr(10).join(_params)
            def fn(**kwargs):
                return format_result(run_tool_spec(spec, dict(kwargs)))
            fn.__name__ = spec['name']
            fn.__doc__ = _desc
            return lc_tool(fn)
        agent.tools.append(_make_runner())
        injected += 1
    else:
        agent.tools.append(w)
        injected += 1
    # Register the raw wrapper into the agent's code-execution REPL namespace
    # (A1 needs this so bqtools_encode(...) is callable inside <execute>).
    if _register_via_add_tool:
        try:
            agent.add_tool(w)
        except Exception as _e:
            display(Markdown(f'`add_tool({getattr(w, "__name__", "tool")})` failed: `{_e}`'))

display(Markdown(f'**{injected} tools injected** into `{type(agent).__name__}.tools`'))

# Show available tool names
tool_names = []
for t in agent.tools:
    name = getattr(t, 'name', None) or getattr(t, '__name__', '?')
    tool_names.append(name)
display(Markdown('**Available tools:** ' + ', '.join(tool_names)))

# Verify exec method
is_known = any(pat in agent_dir.lower() or pat in type(agent).__name__.lower()
               for pat in KNOWN_AGENTS)
if is_known:
    display(Markdown(f'**Known agent** -> method `{exec_method_override}` (trusted)'))
else:
    probe = None
    for m in PROBE_ORDER:
        if m == '__call__':
            if callable(agent): probe = m; break
        elif hasattr(agent, m) and callable(getattr(agent, m)):
            probe = m; break
    if probe:
        display(Markdown(f'**Probe OK:** `agent.{probe}()` exists'))
        get_ipython().user_ns['exec_method_override'] = probe
    else:
        tried = ', '.join(f'`{m}()`' for m in PROBE_ORDER)
        display(Markdown(f'**Probe FAILED:** tried {tried}'))
        display(Markdown('Go to **Step 5b** to provide your agent init code manually.'))

get_ipython().user_ns['agent'] = agent
get_ipython().user_ns['wrappers'] = wrappers
get_ipython().user_ns['adapter'] = adapter

# Detect agent type: tool calling vs code execution
_name_lower = type(agent).__name__.lower()
_use_native = (exec_mode.value == 'Native tool calling')
_is_tool_calling = _use_native and (
    'react' in _name_lower or
    'biochatter' in _name_lower or
    hasattr(agent, 'tool_executor')
)
if _use_native and not _is_tool_calling:
    display(Markdown('**Note:** native tool calling unavailable for this agent; '
                     'falling back to the unified Tool Runner loop.'))
display(Markdown(f'**Exec mode:** `{"Native" if _is_tool_calling else "Unified (Tool Runner)"}` '
                 f'for agent `{type(agent).__name__}`'))

if _is_tool_calling and agent.tools:
    # Tool calling agent: will use agent.tools natively
    _tool_retriever = None
    display(Markdown(f'**Tool calling agent** → {len(agent.tools)} tools ready'))
else:
    # Code execution agent: build retriever for dynamic prompt injection
    from agent_connector.graph_retrieval import build_graph_from_tools, retrieve_tools
    from agent_connector.artifact_spec import annotate_inputs_with_artifacts as _annotate_inputs_with_artifacts
    import yaml as _yaml
    _reg_path = os.path.join('data', 'mcp_registry.yaml')
    if os.path.exists(_reg_path):
        with open(_reg_path, 'r', encoding='utf-8') as _f:
            _reg = _yaml.safe_load(_f) or {}
        _tool_graph = build_graph_from_tools(_reg.get('tools', []))
        # Also add injected tools to the graph
        _injected_dicts = []
        for t in wrappers:
            ts = getattr(t, '_TOOL_SPEC', None)
            if ts:
                _injected_dicts.append(ts)
        if _injected_dicts:
            _injected_graph = build_graph_from_tools(_injected_dicts)
        else:
            _injected_graph = None
        # Build name → spec lookup for prompt generation
        _spec_lookup = {}
        for t in _reg.get('tools', []):
            _spec_lookup[t['name']] = t
            if t.get('subcommand_details'):
                for sub, detail in t['subcommand_details'].items():
                    _fname = f"{t['name']}_{sub.replace('-', '_')}"
                    # Build the leaf from the FULL base-tool spec so execution
                    # metadata (command / arg_style / type / execution) is kept;
                    # only name/description/inputs are overridden per subcommand.
                    _leaf = dict(t)
                    _leaf['name'] = _fname
                    _leaf['description'] = detail.get('description', '')
                    _leaf['inputs'] = _annotate_inputs_with_artifacts(
                        {p['name'].lstrip('-').replace('-', '_'): dict(p)
                         for p in (detail.get('params') or [])},
                        tool_name=t['name'], sub_name=sub)
                    _leaf['_active_subcommand'] = sub
                    if not _leaf.get('execution'):
                        _leaf['execution'] = {'type': _leaf.get('type', 'cli'),
                                              'command': _leaf.get('command', '')}
                    _spec_lookup[_fname] = _leaf

        for t in wrappers:
            ts = getattr(t, '_TOOL_SPEC', None)
            if ts:
                _spec_lookup[ts['name']] = ts
        _tool_retriever = {
            'graph': _tool_graph,
            'injected_graph': _injected_graph,
            'spec_lookup': _spec_lookup,
        }
        # Contract pins: defend against STALE registry/agent_connector data.
        # bqtools_encode CONSUMES FASTA/FASTQ and PRODUCES BINSEQ -- its input is
        # NOT binseq (that base rule applies to decode/revcomp/etc.). If the
        # discovered yaml or artifact_spec is outdated, this forces correctness.
        _enc = _spec_lookup.get('bqtools_encode')
        if _enc and 'input' in (_enc.get('inputs') or {}):
            _ei = _enc['inputs']['input']
            _ei['artifact_type'] = 'fasta'
            _ei['extensions'] = ['.fa', '.fasta', '.fq', '.fastq']
            _ei['required'] = True
        _dec = _spec_lookup.get('bqtools_decode')
        if _dec and 'input' in (_dec.get('inputs') or {}):
            _di = _dec['inputs']['input']
            _di['artifact_type'] = 'binseq'
            _di['extensions'] = ['.vbq', '.cbq', '.bq']
        _rc = _spec_lookup.get('bqtools_revcomp')
        if _rc and 'input' in (_rc.get('inputs') or {}):
            _ri = _rc['inputs']['input']
            _ri['artifact_type'] = 'binseq'
            _ri['extensions'] = ['.vbq', '.cbq', '.bq']
        display(Markdown(f'**Code execution agent** → retriever built ({len(_spec_lookup)} tools indexed)'))

    else:
        _tool_retriever = None
        display(Markdown('**Warning:** mcp_registry.yaml not found, no retrieval'))

get_ipython().user_ns['_tool_retriever'] = _tool_retriever

In [ ]:
from IPython.display import display, Markdown
from agent_connector.agent_runner import build_runtime
import os as _os2

_oai_client = None
try:
    from openai import OpenAI as _OAI
    _oai_client = _OAI(
        api_key=_os2.environ.get('OPENAI_API_KEY') or _os2.environ.get('WESTLAKE_API_KEY'),
        base_url=_os2.environ.get('OPENAI_BASE_URL'),
    )
except Exception:
    _oai_client = None

# Point the MCP server (server.py) at THIS repo's discovered tools + working dir
import os as _os_mcp
_CWD = _os_mcp.getcwd()
_os_mcp.environ.setdefault("MCP_APP_ROOT", _CWD)
_os_mcp.environ.setdefault("MCP_DATA_ROOT", _CWD)
_os_mcp.environ.setdefault("MCP_REGISTRY_PATH", _os_mcp.path.join(_CWD, "registry.yaml"))
_os_mcp.environ.setdefault("MCP_USER_REGISTRY_PATH", _os_mcp.path.join(_CWD, "data", "mcp_registry.yaml"))
_mcp_yaml = (
    "mcp_servers:\n"
    "  bio-mcp:\n"
    "    enabled: true\n"
    "    command:\n"
    "      - python\n"
    "      - server.py\n"
    "    env:\n"
    f"      MCP_APP_ROOT: {_CWD}\n"
    f"      MCP_DATA_ROOT: {_CWD}\n"
    f"      MCP_REGISTRY_PATH: {_os_mcp.path.join(_CWD, "registry.yaml")}\n"
    f"      MCP_USER_REGISTRY_PATH: {_os_mcp.path.join(_CWD, "data", "mcp_registry.yaml")}\n"
    "      MCP_TRANSPORT: stdio\n"
)
with open(_os_mcp.path.join(_CWD, "mcp_config_cluster.yaml"), "w", encoding="utf-8") as _f:
    _f.write(_mcp_yaml)
_runtime_run, _runtime_info = build_runtime(
    tools=tools, schema=schema, agent=agent, agent_dir=agent_dir,
    exec_mode=exec_mode.value,
    tool_retriever=_tool_retriever,
    openai_client=_oai_client,
    model=_os2.environ.get('OPENAI_MODEL') or 'hy3',
    base_url=_os2.environ.get('OPENAI_BASE_URL'),
    api_key=_os2.environ.get('OPENAI_API_KEY') or _os2.environ.get('WESTLAKE_API_KEY'),
)
use_mcp = _runtime_info.get('driver') in ('mcp', 'mcp-native')
mcp_runner = _runtime_run if use_mcp else None
get_ipython().user_ns['mcp_runner'] = mcp_runner
get_ipython().user_ns['runtime_info'] = _runtime_info
_caps = (schema.get('capabilities') or {}).get('mcp') or {}
display(Markdown(f"**Runtime:** `{_runtime_info.get('driver')}` | supports_mcp=`{_runtime_info.get('supports_mcp')}` | mode from capabilities=`{_caps.get('supported')}`"))
display(Markdown(f"**Note:** {_runtime_info.get('note','')}"))


## Step 5b - Manual agent init (only if Step 5 failed)

If Step 5 succeeded, **skip this cell**.

In [ ]:
from IPython.display import display, Markdown
import ipywidgets as widgets

already_ok = is_known
if not already_ok:
    for m in PROBE_ORDER:
        if m == '__call__':
            if callable(agent): already_ok = True; break
        elif hasattr(agent, m) and callable(getattr(agent, m)):
            already_ok = True; break

if already_ok:
    display(Markdown('Step 5 succeeded. **Skipping.**'))
else:
    display(Markdown('### Agent execution method not recognized\n\n'
        '**Tried:** ' + ', '.join(f'`{m}`' for m in PROBE_ORDER) + '\n\n'
        '**Please paste your agent init code below:**'))
    code_area = widgets.Textarea(
        value='# from my_agent import MyAgent\n# agent = MyAgent(model="gpt-4")\n# agent.add_tools(wrappers)\n',
        placeholder='agent = MyAgent(...)',
        layout=widgets.Layout(width='90%', height='150px'))
    method_input = widgets.Dropdown(
        options=['run', 'execute', 'go', 'predict', 'forward', 'invoke', '__call__'],
        value='run', description='Exec method:', layout=widgets.Layout(width='60%'))
    apply_btn = widgets.Button(description='Apply', button_style='primary')
    out = widgets.Output()

    def on_apply(_):
        out.clear_output()
        with out:
            try:
                local_ns = {'wrappers': wrappers, 'agent_dir': agent_dir}
                exec(code_area.value, local_ns)
                new_agent = local_ns.get('agent')
                if new_agent is None:
                    display(Markdown('Error: no `agent` variable found.')); return
                method = method_input.value
                fn = getattr(new_agent, method, None) if method != '__call__' else new_agent
                if fn is None or not callable(fn):
                    display(Markdown(f'Error: `{method}` not callable.')); return
                get_ipython().user_ns['agent'] = new_agent
                get_ipython().user_ns['exec_method_override'] = method
                agent = new_agent
                display(Markdown(f'**Done!** Agent=`{type(new_agent).__name__}` method=`{method}`'))
            except Exception as e:
                display(Markdown(f'Error: `{e}`'))

    apply_btn.on_click(on_apply)
    display(widgets.VBox([code_area, method_input, apply_btn, out]))

## Step 6 - Run tasks via downstream agent

Your query is sent to the agent via `agent.{method}(query)`. The agent uses the injected tools internally.

**Default task:** summarize a FASTA file - this uses `seqmagick_info` (pure-Python, pip-installable, no Rust needed) and runs out of the box in Colab. `bqtools` BINSEQ tools are also discovered, but they require a Rust/`cargo` toolchain; if selected, the runner reports them as unavailable with their install contract instead of failing silently.

In [ ]:
from IPython.display import display, Markdown
import ipywidgets as widgets
import time
import sys
import os
import json as _json
import re as _re

method = exec_method_override
if method == '__call__':
    exec_fn = agent
else:
    exec_fn = getattr(agent, method, None)

if exec_fn is None or not callable(exec_fn):
    display(Markdown(f'ERROR: `agent.{method}()` not callable'))
if _is_tool_calling and agent.tools:
    display(Markdown('**Mode:** Tool calling (agent.tools)'))
else:
    display(Markdown('**Mode:** Retrieval -> Prompt -> Code execution'))

def _build_tool_prompt(specs: list[dict]) -> str:
    """Build prompt block from tool specs."""
    if not specs:
        return ''
    lines = ['You have access to the following tools:', '']
    for s in specs:
        lines.append(f'[Tool: {s["name"]}]')
        lines.append(f'Description: {s.get("description", "")}')
        inputs = s.get('inputs') or {}
        if inputs:
            lines.append('Arguments:')
            for pname, pmeta in inputs.items():
                if pname == 'subcommand':
                    continue
                ptype = (pmeta or {}).get('type', 'string')
                req = 'required' if (pmeta or {}).get('required') else 'optional'
                pdesc = (pmeta or {}).get('description', '')
                lines.append(f'  - {pname} ({ptype}, {req}): {pdesc}')
    lines.append('')
    lines.append('IMPORTANT: You MUST use these tools. Do NOT write your own Python code.')
    lines.append('To call a tool, use <tool_call> JSON format:')
    lines.append('')
    lines.append('<tool_call>')
    lines.append('{"name": "<tool_name_from_list_above>", "arguments": {"<param>": "<value>"}}')
    lines.append('</tool_call>')
    lines.append('')
    lines.append('Rules:')
    lines.append('- Use the exact tool name from the list above.')
    lines.append('- Arguments must match the schema (required/optional, types).')
    lines.append('- Call ONE tool per <tool_call> block.')
    lines.append('- After receiving <tool_result>, continue solving the task.')
    lines.append('- NEVER fabricate, simulate, or guess tool results. Only emit <tool_call> blocks; the real results are returned to you after execution.')
    lines.append('- Do NOT write <observe>, <tool_result>, or any result/observation text yourself; only describe your plan in plain prose.')
    return chr(10).join(lines)

def _parse_tool_calls(text: str) -> list[dict]:
    """Extract tool calls from agent output. Supports both formats:
    1. <tool_call> JSON </tool_call> (generic format)
    2. <execute> Python code </execute> (A1 code execution format)
    """
    calls = []
    # Format 1: <tool_call> JSON </tool_call>
    for m in _re.finditer(r'<tool_call>(.*?)</tool_call>', text, _re.S):
        try:
            calls.append(_json.loads(m.group(1).strip()))
        except _json.JSONDecodeError:
            pass
    # Format 2: <execute> Python code </execute> — extract as raw code block
    if not calls:
        for m in _re.finditer(r'<execute>(.*?)</execute>', text, _re.S):
            code = m.group(1).strip()
            if code:
                calls.append({'_type': 'code', 'code': code})
    return calls

def _execute_tool_call(call: dict) -> str:
    """Execute a parsed tool call. Handles both JSON tool calls and code blocks."""
    from agent_connector.tool_runner import run_tool_spec, format_result
    import subprocess as _sp
    # Code block execution (A1-style <execute>)
    if call.get('_type') == 'code':
        code = call['code']
        try:
            cp = _sp.run([sys.executable, '-c', code], capture_output=True,
                         text=True, timeout=120, encoding='utf-8', errors='replace')
            output = cp.stdout or ''
            if cp.stderr:
                output += '\n[stderr]\n' + cp.stderr
            return output.strip() or '[no output]'
        except Exception as e:
            return f'[error] {type(e).__name__}: {e}'
    # JSON tool call (generic format)
    name = call.get('name', '')
    args = call.get('arguments', {})
    spec = _tool_retriever['spec_lookup'].get(name)
    if not spec:
        return f'[error] Tool "{name}" not found in registry'
    return format_result(run_tool_spec(spec, args))

def _retrieve_and_build_prompt(query: str) -> str:
    """Retrieve relevant tools and build enhanced query."""
    if _tool_retriever is None or not _tool_retriever.get('graph'):
        return query
    # Retrieve from registry
    results = retrieve_tools(_tool_retriever['graph'], query, top_k=20)
    # Also retrieve from injected tools if available
    injected_results = []
    if _tool_retriever.get('injected_graph'):
        injected_results = retrieve_tools(_tool_retriever['injected_graph'], query, top_k=20)
    # Merge: injected tools first, then registry
    _scored = {}
    for name, score, conf in (injected_results or []) + (results or []):
        if name not in _scored or score > _scored[name]:
            _scored[name] = score
    # Tie-break: in this environment cargo tools (bqtools) cannot be
    # installed (no Rust toolchain), so among near-tied retrieval scores
    # prefer tools that ARE installable here (pip/npm). The agent still
    # chooses; this just orders runnable tools first.
    _lookup = _tool_retriever.get('spec_lookup', {})
    def _rank_key(name):
        _m = ((_lookup.get(name) or {}).get('install') or {}).get('method', '')
        _installable = _m in ('pip', 'pip_pkg', 'pip_url', 'npm')
        return (0 if _installable else 1, -_scored.get(name, 0.0))
    top_tools = [n for n in sorted(_scored, key=_rank_key)]
    if not top_tools:
        return query
    # Build prompt with retrieved tool schemas
    specs = [_tool_retriever['spec_lookup'][n] for n in top_tools if n in _tool_retriever['spec_lookup']]
    tool_block = _build_tool_prompt(specs)
    return f'{tool_block}\n\nTask: {query}'

query_input = widgets.Text(
    value='I have a FASTA file reads.fasta. Step 1: summarize reads.fasta (report the number of sequences and the total length in bp). Step 2: create a reverse-complemented FASTA file rc.fasta from reads.fasta. Report the summary from step 1 and the number of sequences in rc.fasta from step 2.',
    placeholder='Ask a bioinformatics question...',
    description='Query:', layout=widgets.Layout(width='95%'))
run_btn = widgets.Button(description='Run agent', button_style='success')
result_out = widgets.Output()

# ---- File upload: saved into ./uploads so tools can read them by name ----
upload = widgets.FileUpload(accept='*', multiple=True, description='Upload:')
upload_out = widgets.Output()

def _on_upload(change):
    with upload_out:
        upload_out.clear_output()
        _d = os.path.join('.', 'uploads')
        os.makedirs(_d, exist_ok=True)
        for _fname, _buf in change['new'].items():
            _data = _buf['content'] if isinstance(_buf, dict) and 'content' in _buf else _buf
            _p = os.path.join(_d, _fname)
            with open(_p, 'wb') as _f:
                _f.write(bytes(_data))
            display(Markdown(f'- saved `{_p}`'))
        if change['new']:
            display(Markdown('Uploaded files are available in your query by name, e.g. `uploads/' + _fname + '`.'))

upload.observe(_on_upload, names='value')

def run_query(_):
    result_out.clear_output()
    with result_out:
        query = query_input.value.strip()
        if not query:
            display(Markdown('**Enter a query**')); return
        display(Markdown(f'**Query:** {query}'))
        t0 = time.time()
        try:
            _mcp_runner = get_ipython().user_ns.get('mcp_runner')
            if _mcp_runner is not None:
                display(Markdown('**MCP mode:** driving tools through server.py over MCP'))
                final = _mcp_runner(query)
            elif _is_tool_calling and agent.tools:
                # Direct tool calling path (react, BioChatter)
                display(Markdown(f'**Calling:** `agent.{method}(query)` ...'))
                final = exec_fn(query)
            else:
                # Code execution agent: Retrieval → LLM API → Parse → Execute → Feed back
                enhanced_query = _retrieve_and_build_prompt(query)
                # Use agent's own LLM — keeps agent alive, bypasses A1's broken sandbox
                from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
                _llm = getattr(agent, 'llm', None)
                if _llm is None:
                    # Fallback: create OpenAI client directly
                    from openai import OpenAI as _OAI
                    _key = get_ipython().user_ns.get('api_key_val', '')
                    _base = get_ipython().user_ns.get('base_url_val', '')
                    _model = get_ipython().user_ns.get('model_val', 'hy3')
                    _oai = _OAI(api_key=_key, base_url=_base)
                    _llm = None
                else:
                    _oai = None
                # Build messages: system prompt from agent + user query
                _sys = getattr(agent, 'system_prompt', '') or getattr(agent, '_system_prompt', '') or ''
                if _llm is not None:
                    lc_messages = []
                    if _sys:
                        lc_messages.append(SystemMessage(content=_sys))
                    lc_messages.append(HumanMessage(content=enhanced_query))
                else:
                    messages = []
                    if _sys:
                        messages.append({"role": "system", "content": _sys})
                    messages.append({"role": "user", "content": enhanced_query})
                MAX_ITER = 5
                final = ''
                _done = []
                for iteration in range(MAX_ITER):
                    display(Markdown('Working…'))
                    if _llm is not None:
                        resp = _llm.invoke(lc_messages)
                        raw_str = resp.content if hasattr(resp, 'content') else str(resp)
                        lc_messages.append(AIMessage(content=raw_str))
                    else:
                        resp = _oai.chat.completions.create(
                            model=_model, messages=messages, temperature=0.3)
                        raw_str = resp.choices[0].message.content or ''
                        messages.append({"role": "assistant", "content": raw_str})
                    # Parse tool calls from LLM response
                    calls = _parse_tool_calls(raw_str)
                    if not calls:
                        final = raw_str
                        break
                    tool_results = []
                    for c in calls:
                        name = c.get('name', c.get('_type', '?'))
                        display(Markdown(f'**Running tool:** `{name}` …'))
                        result_str = _execute_tool_call(c)
                        tool_results.append(f'<tool_result>\n{result_str}\n</tool_result>')
                        _akey = _json.dumps(c.get('arguments', {}) or {}, sort_keys=True)
                        _ok = ('error' not in result_str.lower()) and ('command_error' not in result_str.lower())
                        _done.append((name, _akey, _ok))
                        if not _ok:
                            display(Markdown(f'  [FAIL] `{name}` failed'))
                    # Feed results back to LLM (with anti-loop + chaining hints)
                    feedback = '\n'.join(tool_results)
                    _summary = '\n'.join(f'- {nm} (args={akey}): {"SUCCESS" if ok else "FAILED"}' for (nm, akey, ok) in _done)
                    _loop_note = ('\n\nAlready executed this session \u2014 do NOT repeat any of them;'
                                  ' chain to the NEXT step using the output they produced:'
                                  '\n' + _summary) if _done else ''
                    feedback_msg = f'Tool execution results:\n{feedback}{_loop_note}\n\nContinue with the next step or provide the final answer.'
                    feedback = '\n'.join(tool_results)
                    feedback_msg = f'Tool execution results:\n{feedback}\n\nContinue with the next step or provide the final answer.'
                    if _llm is not None:
                        lc_messages.append(HumanMessage(content=feedback_msg))
                    else:
                        messages.append({"role": "user", "content": feedback_msg})
                else:
                    final = raw_str if 'raw_str' in dir() else '(max iterations reached)'
            elapsed = time.time() - t0
            display(Markdown(f'**Done** ({elapsed:.1f}s)'))
            display(Markdown(f'### Result\n{str(final)[:3000]}'))
        except Exception as e:
            display(Markdown(f'**Error:** `{type(e).__name__}: {e}`'))

run_btn.on_click(run_query)
display(widgets.VBox([query_input, run_btn, upload, upload_out, result_out]))